# Magnitude MP-PCA Denoising

Gaussian noise가 추가된 multi-echo GRE complex 데이터를 magnitude로 변환한 뒤, DIPY의 MP-PCA를 적용하여 denoising을 수행한다.

10%, 30%, 50% noise 데이터에 동일한 조건의 MP-PCA를 적용하여 noise level에 따른 denoising 결과를 비교한다.

In [32]:
import numpy as np
import scipy.io as sio
from pathlib import Path
import matplotlib.pyplot as plt
from dipy.denoise.localpca import mppca

### 환경설정

Magnitude MP-PCA 실험에 사용할 기본 조건을 설정한다.

- Noise level: 10%, 30%, 50%
- Patch radius: 2
- Patch size: 5 × 5 × 5
- 입력 데이터 경로: `./data/noisy_data`
- 결과 저장 경로: `./data/denoised/magnitude`

`PATCH_R = 2`이면 중심 voxel을 기준으로 각 방향으로 2 voxel씩 포함하므로 5 × 5 × 5 크기의 local patch가 사용된다.

모든 noise level에 동일한 patch 크기를 적용하여 MP-PCA 조건을 동일하게 유지한다.

### 1. Noise 데이터 불러오기

`Add_noise` 단계에서 생성한 10%, 30%, 50% noisy `.mat` 파일을 각각 불러온다.

사용하는 파일은 다음과 같다.

- `noisy_meas_gre_dir1_10.mat`
- `noisy_meas_gre_dir1_30.mat`
- `noisy_meas_gre_dir1_50.mat`

각 파일에는 noise가 추가된 complex MRI 데이터인 `noisy_complex`와 brain mask, MRI 메타데이터가 포함되어 있다.

불러온 데이터는 noise level을 기준으로 구분하여 저장하고, 이후 magnitude 변환과 denoising 과정에서 사용한다.

데이터를 불러온 후에는 `noisy_complex`의 shape가 원본 데이터와 동일한지 확인한다.

### 2. Magnitude 변환

MP-PCA를 적용하기 전에 noisy complex 데이터를 magnitude 영상으로 변환한다.

Complex MRI 데이터는 실수부와 허수부를 모두 포함하고 있으며, Magnitude 방식에서는 두 성분으로부터 신호의 크기만 계산하여 사용한다.

이 과정에서는 phase 정보는 사용하지 않는다.

10%, 30%, 50% noise 데이터를 각각 magnitude로 변환한 뒤 이후 MP-PCA 입력 데이터로 사용한다.

변환 후에는 데이터 shape가 기존 complex 데이터와 동일한지 확인한다.

### 3. MP-PCA Denoising

각 noise level의 magnitude 데이터에 DIPY의 `mppca()`를 적용한다.

MP-PCA는 주변 voxel로 구성된 local patch에서 PCA를 수행하고, noise 성분으로 판단되는 principal component를 제거하여 영상을 복원하는 방식이다.

본 실험에서는 다음 조건을 동일하게 적용한다.

- Patch radius: 2
- Patch size: 5 × 5 × 5
- Brain mask 내부 영역에 대해서만 MP-PCA 수행

10%, 30%, 50% 데이터에 동일한 patch 크기를 사용하여 noise level에 따른 denoising 성능을 비교할 수 있도록 한다.

Denoising 완료 후 결과가 각 noise level별로 정상적으로 생성되었는지 확인한다.

### 4. 결과 저장

각 noise level의 Magnitude MP-PCA 결과를 별도의 `.mat` 파일로 저장한다.

저장 파일은 다음과 같다.

- `denoised_magnitude_10_r2.mat`
- `denoised_magnitude_30_r2.mat`
- `denoised_magnitude_50_r2.mat`

파일명의 `r2`는 MP-PCA에서 사용한 patch radius가 2임을 의미한다.

각 결과 파일에는 denoising 결과와 함께 brain mask, noise level, patch radius 및 MRI 메타데이터를 저장한다.

원본 MRI 데이터와 noisy 데이터는 기존 파일에 별도로 보관되어 있으므로 denoising 결과 파일에는 중복 저장하지 않는다.

### 5. 시각화

Magnitude MP-PCA가 정상적으로 적용되었는지 확인하기 위해 원본, noisy, denoised 영상을 동일한 조건에서 비교한다.

본 실험에서는 예시로 다음 위치를 사용한다.

- Echo: 1
- Slice: 88

각 noise level에 대해 다음 네 가지 영상을 비교한다.

- Original: noise가 추가되기 전 원본 magnitude 영상
- Noisy: Gaussian noise가 추가된 magnitude 영상
- Denoised: MP-PCA 적용 후 magnitude 영상
- Difference: Denoised 영상과 Original 영상의 차이

Noisy 영상과 Denoised 영상을 비교하여 noise가 감소했는지 확인하고, Difference 영상을 통해 원본 신호가 과도하게 변화하거나 손실되지 않았는지 확인한다.

### 최종 확인사항

Magnitude MP-PCA 실험이 완료되면 다음 사항을 확인한다.

- 10%, 30%, 50% noisy 데이터가 정상적으로 불러와졌는지 확인
- Complex 데이터가 magnitude로 정상 변환되었는지 확인
- 모든 noise level에 동일한 5 × 5 × 5 patch가 적용되었는지 확인
- Brain mask가 정상적으로 적용되었는지 확인
- 10%, 30%, 50% denoising 결과가 모두 생성되었는지 확인
- 결과 `.mat` 파일이 각각 저장되었는지 확인
- Original / Noisy / Denoised 영상이 정상적으로 표시되는지 확인
- Denoising 후 noise가 감소하는지 확인
- Difference 영상에서 과도한 신호 손실이 나타나지 않는지 확인

이 결과는 이후 SNR, SSIM 등의 정량적 평가와 다른 MP-PCA 방식의 결과 비교에 사용할 수 있다.